In [ ]:
import rasterio

# 파일 경로
tif_path = r'<DATA_ROOT>\05_smelter\Germany\03_Dillengen\05_result\per_date\S2A_MSIL1C_20160106T103422_N0500_R108_T31UGQ_20231009T024937_condition_mapped.tif'

# 열, 행 순서로 지정
col = 249
row = 280

with rasterio.open(tif_path) as src:
    # 픽셀 값 추출 (밴드 1 기준)
    pixel_value = src.read(1)[row, col]  # numpy 배열이므로 (행, 열) 순서
    print(f"픽셀 값 (행 {row}, 열 {col}): {pixel_value}")


In [ ]:
import rasterio

# 파일 경로
tif_path = r'<DATA_ROOT>\05_smelter\Germany\03_Dillengen\FALSE\S2A_MSIL1C_20160408T104022_N0500_R008_T31UGQ_20231003T214711_reduced_infraredRGB.tif'

# 열, 행 순서로 지정
col = 251
row = 280

with rasterio.open(tif_path) as src:
    band_count = src.count  # 총 밴드 수 확인
    print(f"총 밴드 수: {band_count}")
    
    for b in range(1, band_count + 1):  # 밴드는 1부터 시작
        pixel_value = src.read(b)[row, col]
        print(f"밴드 {b} - 픽셀 값 (행 {row}, 열 {col}): {pixel_value}")


In [ ]:
# TAI 계산 코드

import numpy as np
import rasterio
from scipy.signal import convolve2d
import os
from tqdm import tqdm  # tqdm 추가

# 입력 폴더 경로 (여러 tif 파일 포함)
tif_dir = r"E:\Smelter\Germany\05_Bremen\FALSE"
# 출력 폴더 경로
output_dir = r"E:\Smelter\Germany\05_Bremen\TAI\0_05"

# 파라미터 설정
thresholdTAI = 0.05
radius = 15

# AOI 마스크 함수 (전체 사용)
def get_aoi_mask(shape):
    return np.ones(shape, dtype=bool)

# 디렉토리 생성
os.makedirs(output_dir, exist_ok=True)

# tif 파일 목록 가져오기
tif_files = [f for f in os.listdir(tif_dir) if f.endswith('.tif')]

# tqdm 적용
for filename in tqdm(tif_files, desc="Processing TIF files"):
    input_path = os.path.join(tif_dir, filename)
    output_filename = os.path.splitext(filename)[0] + f"_TAI_{thresholdTAI:.2f}.tif"
    output_path = os.path.join(output_dir, output_filename)

    with rasterio.open(input_path) as src:
        B12 = src.read(1).astype(np.float32)
        B11 = src.read(2).astype(np.float32)
        B8A = src.read(3).astype(np.float32)
        profile = src.profile

    # TAI 계산
    TAI = (B12 - B11) / B8A
    TAI[~np.isfinite(TAI)] = 0
    TAI[TAI < 0] = 0

    # 평균 필터
    ky, kx = np.ogrid[-radius:radius+1, -radius:radius+1]
    mask = kx**2 + ky**2 <= radius**2
    kernel = mask.astype(float)
    kernel /= kernel.sum()
    TAI_mean = convolve2d(TAI, kernel, mode='same', boundary='fill')
    TAI_mean_whole = TAI_mean.copy()

    # AOI 마스크 적용
    cond_AOI20m = get_aoi_mask(TAI.shape)
    TAI[~cond_AOI20m] = np.nan
    TAI_mean[~cond_AOI20m] = 0

    # 필터링
    TAI[(TAI - TAI_mean < thresholdTAI)] = 0
    TAI[(B12 - B11 <= B11 - B8A)] = 0
    TAI[B12 <= 0.15] = 0
    TAI[B11 <= 0.05] = 0
    TAI[B8A <= 0.01] = 0

    # 저장 준비
    TAI_to_save = np.nan_to_num(TAI, nan=0).astype(np.float32)
    profile.update({
        'count': 1,
        'dtype': 'float32',
        'compress': 'lzw'
    })

    # 저장
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(TAI_to_save, 1)


In [ ]:
"""
Sentinel-2 영상에서 B12 ≥ 1 & B11 ≥ 1인 픽셀 중,
동일 날짜의 TAI 영상에서 값이 0 또는 NODATA인 경우에만 주변 (3x3→5x5→7x7) TAI max 값으로 보간.
기존 TAI > 0 픽셀은 그대로 유지하여 새로운 TAImax 영상을 생성하는 자동화 스크립트.
"""

import os
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm

# --------------------- 사용자 정의 ---------------------
region = "Eisenhuttenstadt"  # 지역 이름만 입력 (숫자 제외)
base_dir = r"<DATA_ROOT>\05_smelter\Germany"
radius_list = [1, 2, 3]  # 보간 창 크기: 3x3 → 5x5 → 7x7
# -------------------------------------------------------

# 지역명이 포함된 폴더 자동 탐색
candidate_dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and region in d]
if not candidate_dirs:
    raise FileNotFoundError(f"'{region}'을 포함한 폴더가 {base_dir} 안에 없습니다.")
region_code = candidate_dirs[0]  # 첫 번째 매칭 폴더 사용

# 경로 설정
false_dir = os.path.join(base_dir, region_code, "FALSE")
tai_dir = os.path.join(base_dir, region_code, "TAI", "0_05")
output_dir = os.path.join(base_dir, region_code, "TAImax")
csv_path = os.path.join(base_dir, "08_dates", "Germany",
                        f"earthiGT_SentinelLandsat_measurements_{region}_BFsSlag_yr2018-2023_noForwardFilling.csv")

# 저장 폴더 생성
os.makedirs(output_dir, exist_ok=True)

# 날짜 CSV 로드
df = pd.read_csv(csv_path)
dates_csv = pd.to_datetime(df.iloc[:, 0], format="%Y-%m-%d").dt.strftime("%Y%m%d").tolist()

# 파일명에서 날짜 추출
def extract_date_from_filename(filename):
    try:
        return filename.split('_')[2][:8]
    except IndexError:
        return ""

# 보간 함수
def get_max_in_window(arr, y, x, radius_list):
    h, w = arr.shape
    for r in radius_list:
        y_min, y_max = max(0, y - r), min(h, y + r + 1)
        x_min, x_max = max(0, x - r), min(w, x + r + 1)
        window = arr[y_min:y_max, x_min:x_max]
        valid = window[window > 0]
        if valid.size > 0:
            return np.max(valid)
    return 0.0

# 처리 대상 TIF 파일 리스트
tif_files = [
    f for f in os.listdir(false_dir)
    if f.endswith(".tif") and extract_date_from_filename(f) in dates_csv
]

# 메인 처리 루프
for filename in tqdm(tif_files, desc=f"Processing {region} ({region_code})"):
    date_str = extract_date_from_filename(filename)
    false_path = os.path.join(false_dir, filename)

    # 해당 날짜의 TAI 파일 찾기
    tai_candidates = [f for f in os.listdir(tai_dir) if date_str in f and f.endswith(".tif")]
    if not tai_candidates:
        print(f"❌ TAI 파일 없음: {filename}")
        continue
    tai_path = os.path.join(tai_dir, tai_candidates[0])

    # 출력 파일 경로 설정
    output_filename = os.path.splitext(filename)[0] + "_TAImax_interpolated.tif"
    output_path = os.path.join(output_dir, output_filename)

    # 파일 읽기
    with rasterio.open(false_path) as src_false, rasterio.open(tai_path) as src_tai:
        B12 = src_false.read(1).astype(np.float32)
        B11 = src_false.read(2).astype(np.float32)
        TAI = src_tai.read(1).astype(np.float32)
        profile = src_false.profile
        nodata_val = src_tai.nodata

    # 결과 초기화: 기존 TAI 값 그대로 유지
    result = np.copy(TAI)

    # 보간 대상: B12≥1, B11≥1, 그리고 TAI 값이 0 또는 NODATA
    if nodata_val is not None:
        target_mask = (B12 >= 1) & (B11 >= 1) & ((TAI == 0) | (TAI == nodata_val))
    else:
        target_mask = (B12 >= 1) & (B11 >= 1) & ((TAI == 0) | np.isnan(TAI))

    ys, xs = np.where(target_mask)

    # 보간 수행
    for y, x in zip(ys, xs):
        result[y, x] = get_max_in_window(TAI, y, x, radius_list)

    # 저장
    profile.update({
        'count': 1,
        'dtype': 'float32',
        'compress': 'lzw'
    })
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(result, 1)

print(f"\n✅ {region_code} 지역의 TAImax 보간 완료!\n결과 저장 경로: {output_dir}")


지역마다 TAI 합산 값 계산

In [ ]:
import os
import glob
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from datetime import datetime

# ------------------ 사용자 설정 ------------------
region_name = 'TK'
shp_path = fr'<DATA_ROOT>\05_smelter\Germany\shp\01_unzipped\{region_name}_Steel_Duisburg_Blast_Furnaces\{region_name}_Steel_Duisburg_Blast_Furnaces.shp'
tif_dir = fr'<DATA_ROOT>\05_smelter\Germany\01_{region_name}\TAImax'
output_csv = fr'<DATA_ROOT>\05_smelter\Germany\09_sum\TAImax_monthly_sum_{region_name}.csv'

# ------------------ Shapefile 읽기 ------------------
gdf = gpd.read_file(shp_path)

# ------------------ 월별 합계 계산 ------------------
monthly_sums = {}
tif_files = glob.glob(os.path.join(tif_dir, '*.tif'))

for tif_path in tif_files:
    filename = os.path.basename(tif_path)

    # 예: 'S2A_MSIL1C_20180108T104421_...' → '20180108'
    try:
        date_str = filename.split('_')[2][:8]
        date = datetime.strptime(date_str, '%Y%m%d')
        year_month = date.strftime('%Y-%m')
    except (IndexError, ValueError):
        print(f"[날짜 추출 실패] {filename}")
        continue

    # TIF 열기 및 마스킹
    with rasterio.open(tif_path) as src:
        try:
            # 좌표계를 TIF에 맞춰 변환
            gdf_proj = gdf.to_crs(src.crs)

            out_image, _ = mask(src, gdf_proj.geometry, crop=True)
            data = out_image[0]

            # NODATA 제외
            data = data[data != src.nodata]
            if data.size == 0:
                continue

            pixel_sum = data.sum()
        except Exception as e:
            print(f"[마스킹 실패] {filename} → {e}")
            continue

    # 월별 누적합
    monthly_sums[year_month] = monthly_sums.get(year_month, 0) + pixel_sum

# ------------------ 결과 저장 ------------------
df = pd.DataFrame(sorted(monthly_sums.items()), columns=['Month', 'Sum'])
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df.to_csv(output_csv, index=False)

print(f'[완료] CSV 저장: {output_csv}')
